# Model selection — evaluation and comparison

Loads `models/model_selection_outputs/evaluation_pack.pkl` produced by
[`03-model-selection-advanced.ipynb`](03-model-selection-advanced.ipynb) (last cell).

**Recommended run order**
1. [`01-model-selection-exploration.ipynb`](01-model-selection-exploration.ipynb) — optional; Part 1 CV numbers in the pack if run before advanced.
2. [`02-model-selection-baseline.ipynb`](02-model-selection-baseline.ipynb) — optional leak-free CV grid only.
3. [`03-model-selection-advanced.ipynb`](03-model-selection-advanced.ipynb) — required; writes the pack.

If Part 1 was skipped, the Part 1 row prints as unavailable.


In [1]:
import pickle

from msa.utils.paths import get_model_selection_outputs_path

OUT_DIR = get_model_selection_outputs_path()
pack_path = OUT_DIR / 'evaluation_pack.pkl'
if not pack_path.exists():
    raise FileNotFoundError(f'Missing {pack_path}. Run 03-model-selection-advanced.ipynb to the end first.')

with open(pack_path, 'rb') as f:
    pack = pickle.load(f)

print('=' * 70)
print('FINAL COMPARISON — did we hit 65%?')
print('=' * 70)
print()
print(f"{'approach':<50} {'AUC':>8} {'ACC':>8}")
print('-' * 70)

p1 = pack.get('part1_beat_best')
if p1:
    print(f"{'part 1: best baseline (beat_market)':<50} {p1['auc']:>8.4f} {p1['acc']:>8.4f}")
else:
    print(f"{'part 1: best baseline (beat_market) — run 01 exploration':<50} {'n/a':>8} {'n/a':>8}")

p2 = pack['part2_beat_best']
print(f"{'part 2: best leak-free model':<50} {p2['auc']:>8.4f} {p2['acc']:>8.4f}")
print(f"{'part 3: Optuna XGBoost (expanded features)':<50} {pack['part3_optuna_xgb']['auc']:>8.4f} {pack['part3_optuna_xgb']['acc']:>8.4f}")
print(f"{'part 3: Optuna LightGBM (expanded features)':<50} {pack['part3_optuna_lgbm']['auc']:>8.4f} {pack['part3_optuna_lgbm']['acc']:>8.4f}")
print(f"{'part 3: Ensemble (XGB + LGBM averaged)':<50} {pack['part3_ensemble_auc']:>8.4f} {pack['part3_ensemble_acc']:>8.4f}")

if pack.get('part3_per_ticker_auc') is not None:
    print(f"{'part 3: Per-ticker Optuna XGBoost':<50} {pack['part3_per_ticker_auc']:>8.4f} {pack['part3_per_ticker_acc']:>8.4f}")

print(f"{'part 3: Walk-forward (Optuna XGB, expanded)':<50} {pack['part3_walk_forward_auc']:>8.4f} {pack['part3_walk_forward_acc']:>8.4f}")
print()
print('=' * 70)

all_aucs = {
    'p2_leak_free': p2['auc'],
    'p3_optuna_xgb': pack['part3_optuna_xgb']['auc'],
    'p3_optuna_lgbm': pack['part3_optuna_lgbm']['auc'],
    'p3_ensemble': pack['part3_ensemble_auc'],
    'p3_walk_forward': pack['part3_walk_forward_auc'],
}
if p1:
    all_aucs['p1_baseline'] = p1['auc']
if pack.get('part3_per_ticker_auc') is not None:
    all_aucs['p3_per_ticker'] = pack['part3_per_ticker_auc']

best_name = max(all_aucs, key=all_aucs.get)
best_auc = all_aucs[best_name]

if best_auc >= 0.65:
    print(f'WE HIT 65%! best AUC = {best_auc:.4f} ({best_name})')
elif best_auc >= 0.60:
    print(f'close! best AUC = {best_auc:.4f} ({best_name})')
    print(f"  gap to 65%: {(0.65 - best_auc)*100:.1f} percentage points")
    print('  for MAG7 liquid large-caps, AUC > 0.60 is already a strong result')
else:
    print(f'best AUC = {best_auc:.4f} ({best_name})')
    print(f"  gap to 65%: {(0.65 - best_auc)*100:.1f} percentage points")

print()
print('what each improvement added:')
if p1:
    print(f"  part 1 -> part 2 (leak fixes):         {(p2['auc'] - p1['auc'])*100:+.1f}pp AUC")
    print(f"  part 2 -> part 3 (best vs p2 row):      {(best_auc - p2['auc'])*100:+.1f}pp AUC")
    print(f"  total (p1 best -> overall best):         {(best_auc - p1['auc'])*100:+.1f}pp AUC")
else:
    print('  (Part 1 not in pack — run 01 exploration before advanced for p1 deltas)')

print()
print('remaining ideas if we still need more:')
print('  - add intraday features (open-to-close range, gap up/down)')
print('  - add earnings calendar features (days until next earnings)')
print('  - add options data (put/call ratio, implied vol)')
print('  - use FinBERT on article titles only (less noise than full text)')
print('  - try CatBoost (handles categoricals natively)')
print('  - add cross-sectional features (ticker return rank within MAG7)')


FINAL COMPARISON — did we hit 65%?

approach                                                AUC      ACC
----------------------------------------------------------------------
part 1: best baseline (beat_market)                  0.4850   0.4604
part 2: best leak-free model                         0.5000   0.5321
part 3: Optuna XGBoost (expanded features)           0.5000   0.5500
part 3: Optuna LightGBM (expanded features)          0.4138   0.4375
part 3: Ensemble (XGB + LGBM averaged)               0.4138   0.4375
part 3: Walk-forward (Optuna XGB, expanded)          0.4183   0.4831

best AUC = 0.5000 (p2_leak_free)
  gap to 65%: 15.0 percentage points

what each improvement added:
  part 1 -> part 2 (leak fixes):         +1.5pp AUC
  part 2 -> part 3 (best vs p2 row):      +0.0pp AUC
  total (p1 best -> overall best):         +1.5pp AUC

remaining ideas if we still need more:
  - add intraday features (open-to-close range, gap up/down)
  - add earnings calendar features (days until ne